# THE QUANT — Kaggle GPU Agent (free, data-loss-proof)

Setup once:
1. **Settings → Internet → ON** (required)
2. **Settings → Accelerator → GPU T4/P100**
3. **Add-ons → Secrets** → add `GITHUB_TOKEN` (a GitHub fine-grained PAT with Contents read+write on THE-QUANT)
4. Run the cell below — it joins the same job queue as the Colab agent.

From your machine:
```
python3 colab/colab_cli.py enqueue --script colab/jobs/download_data.py --params-json '{"symbols":"eurusd,gbpusd,xauusd","start":"2025-01-01"}'
python3 colab/colab_cli.py status      # watch the queue
python3 colab/colab_cli.py pull        # recover all outputs (any time)
```

In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/THE-QUANT'
# re-run safe: reuse existing clone, heal broken ones, clone if missing
if os.path.exists(REPO + '/.git'):
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', '--quiet'],
                   capture_output=True)
elif os.path.exists(REPO):  # broken/partial checkout -> move aside
    import time, shutil
    shutil.move(REPO, REPO + '.broken-' + time.strftime('%Y%m%d-%H%M%S'))
    subprocess.run(['git', 'clone',
                    'https://github.com/elmaxadore/THE-QUANT.git', REPO],
                   check=True)
else:
    subprocess.run(['git', 'clone',
                    'https://github.com/elmaxadore/THE-QUANT.git', REPO],
                   check=True)

from kaggle_secrets import UserSecretsClient
os.environ['GITHUB_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN')
os.environ['PYTHONUNBUFFERED'] = '1'

subprocess.call('pip install -q torch onnx onnxruntime xgboost pandas numpy requests', shell=True)
print('[+] Joining THE-QUANT job queue... (re-running this cell is safe)')
subprocess.call([sys.executable, REPO + '/colab/agent.py', '--interval', '30'],
                cwd=REPO, env=os.environ)